# Пошаговая очистка
Каждый этап импортируется из src/cleaning.py. Те же функции вызывает src/pipeline.py.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display
from src.config import INPUT, INTERIM, PROCESSED
print(sys.version)
print("Python:", sys.executable)
from src.cleaning import STEPS, apply_step
df = pd.read_parquet(INTERIM / "source_typed.parquet")
audit = []

3.12.14 (main, Aug 25 2026, 14:01:42) [MSC v.1944 64 bit (AMD64)]
Python: <project>\.venv\Scripts\python.exe


## 1. Единые имена столбцов
CamelCase переводится в нижний регистр с подчёркиванием.

In [2]:
df, stats = apply_step(df, STEPS[0])
audit.append(stats)
display(pd.DataFrame([stats]))
display(df.head(2))

,stage,processed,after,deleted,changed,note
0,names,1600000,1600000,0,0,Переименованы столбцы


,id,type,language,license,rights_holder,access_rights,dataset_id,institution_code,dataset_name,owner_institution_code,...,decimal_latitude,decimal_longitude,geodetic_datum,coordinate_uncertainty_in_meters,georeference_sources,georeference_verification_status,scientific_name,kingdom,taxon_rank,vernacular_name
0,801:20130517200539,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,51.233483,2.930108,WGS84,6,GPS,unverified,Larus argentatus,Animalia,species,Herring Gull
1,801:20130517223523,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,51.233446,2.930020,WGS84,14,GPS,unverified,Larus argentatus,Animalia,species,Herring Gull


## 2. Явные типы и даты UTC
Широта и долгота остаются float64, чтобы не терять точность GPS.

In [3]:
df, stats = apply_step(df, STEPS[1])
audit.append(stats)
display(pd.DataFrame([stats]))
display(df.head(2))

,stage,processed,after,deleted,changed,parse_failures
0,assign_types,1600000,1600000,0,0,0


,id,type,language,license,rights_holder,access_rights,dataset_id,institution_code,dataset_name,owner_institution_code,...,decimal_latitude,decimal_longitude,geodetic_datum,coordinate_uncertainty_in_meters,georeference_sources,georeference_verification_status,scientific_name,kingdom,taxon_rank,vernacular_name
0,801:20130517200539,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,51.233483,2.930108,WGS84,6,GPS,unverified,Larus argentatus,Animalia,species,Herring Gull
1,801:20130517223523,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,51.233446,2.930020,WGS84,14,GPS,unverified,Larus argentatus,Animalia,species,Herring Gull


## 3. Скрытые пропуски
Проверяются пустые строки, прочерк, null и нет данных. Нулевая высота не объявляется пропуском.

In [4]:
df, stats = apply_step(df, STEPS[2])
audit.append(stats)
display(pd.DataFrame([stats]))
display(df.head(2))

,stage,processed,after,deleted,changed
0,hidden_missing,1600000,1600000,0,0


,id,type,language,license,rights_holder,access_rights,dataset_id,institution_code,dataset_name,owner_institution_code,...,decimal_latitude,decimal_longitude,geodetic_datum,coordinate_uncertainty_in_meters,georeference_sources,georeference_verification_status,scientific_name,kingdom,taxon_rank,vernacular_name
0,801:20130517200539,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,51.233483,2.930108,WGS84,6,GPS,unverified,Larus argentatus,Animalia,species,Herring Gull
1,801:20130517223523,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,51.233446,2.930020,WGS84,14,GPS,unverified,Larus argentatus,Animalia,species,Herring Gull


## 4. Пропуски обязательных и дополнительных полей
Строки без ключа, даты и координат непригодны. Пол и высота могут быть неизвестны.

In [5]:
df, stats = apply_step(df, STEPS[3])
audit.append(stats)
display(pd.DataFrame([stats]))
display(df.head(2))

,stage,processed,after,deleted,changed
0,missing,1600000,1600000,0,201


,id,type,language,license,rights_holder,access_rights,dataset_id,institution_code,dataset_name,owner_institution_code,...,geodetic_datum,coordinate_uncertainty_in_meters,georeference_sources,georeference_verification_status,scientific_name,kingdom,taxon_rank,vernacular_name,height_missing,sex_missing
0,801:20130517200539,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,WGS84,6,GPS,unverified,Larus argentatus,Animalia,species,Herring Gull,False,False
1,801:20130517223523,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,WGS84,14,GPS,unverified,Larus argentatus,Animalia,species,Herring Gull,False,False


## 5. Полные повторы и повторы ключа
Сохраняется позднее событие; при равенстве дат последняя строка. Даты изменения нет.

In [6]:
df, stats = apply_step(df, STEPS[4])
audit.append(stats)
display(pd.DataFrame([stats]))
display(df.head(2))

,stage,processed,after,deleted,changed,full_duplicates,key_duplicates
0,duplicates,1600000,1600000,0,0,0,0


,id,type,language,license,rights_holder,access_rights,dataset_id,institution_code,dataset_name,owner_institution_code,...,geodetic_datum,coordinate_uncertainty_in_meters,georeference_sources,georeference_verification_status,scientific_name,kingdom,taxon_rank,vernacular_name,height_missing,sex_missing
0,801:20130517200539,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,WGS84,6,GPS,unverified,Larus argentatus,Animalia,species,Herring Gull,False,False
1,801:20130517223523,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,WGS84,14,GPS,unverified,Larus argentatus,Animalia,species,Herring Gull,False,False


## 6. Категории
Регистр меняется только в технических категориях. Названия видов и имена не искажаются. Редкие виды сохраняются.

In [7]:
df, stats = apply_step(df, STEPS[5])
audit.append(stats)
display(pd.DataFrame([stats]))
display(df.head(2))

,stage,processed,after,deleted,changed
0,categories,1600000,1600000,0,0


,id,type,language,license,rights_holder,access_rights,dataset_id,institution_code,dataset_name,owner_institution_code,...,geodetic_datum,coordinate_uncertainty_in_meters,georeference_sources,georeference_verification_status,scientific_name,kingdom,taxon_rank,vernacular_name,height_missing,sex_missing
0,801:20130517200539,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,WGS84,6,GPS,unverified,Larus argentatus,Animalia,species,Herring Gull,False,False
1,801:20130517223523,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,WGS84,14,GPS,unverified,Larus argentatus,Animalia,species,Herring Gull,False,False


## 7. Свободный текст имени птицы
organism_name выполняет роль TEXT. Удаляются HTML, повторные пробелы и служебные значения, если они встречаются.

In [8]:
df, stats = apply_step(df, STEPS[6])
audit.append(stats)
display(pd.DataFrame([stats]))
display(df.head(2))

,stage,processed,after,deleted,changed
0,text_field,1600000,1600000,0,0


,id,type,language,license,rights_holder,access_rights,dataset_id,institution_code,dataset_name,owner_institution_code,...,geodetic_datum,coordinate_uncertainty_in_meters,georeference_sources,georeference_verification_status,scientific_name,kingdom,taxon_rank,vernacular_name,height_missing,sex_missing
0,801:20130517200539,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,WGS84,6,GPS,unverified,Larus argentatus,Animalia,species,Herring Gull,False,False
1,801:20130517223523,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,WGS84,14,GPS,unverified,Larus argentatus,Animalia,species,Herring Gull,False,False


## 8. Единицы измерения и предметные ограничения
Метры и градусы уже согласованы. Отрицательная высота помечается; корректные координаты не удаляются из-за ошибки высоты.

In [9]:
df, stats = apply_step(df, STEPS[7])
audit.append(stats)
display(pd.DataFrame([stats]))
display(df.head(2))

,stage,processed,after,deleted,changed,invalid_domain,negative_height,missing_interval
0,constraints,1600000,1600000,0,363717,0,363699,22


,id,type,language,license,rights_holder,access_rights,dataset_id,institution_code,dataset_name,owner_institution_code,...,georeference_sources,georeference_verification_status,scientific_name,kingdom,taxon_rank,vernacular_name,height_missing,sex_missing,height_negative,seconds_since_previous
0,801:20130517200539,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,GPS,unverified,Larus argentatus,Animalia,species,Herring Gull,False,False,False,1815
1,801:20130517223523,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,GPS,unverified,Larus argentatus,Animalia,species,Herring Gull,False,False,False,1790


## 9. Выбросы по IQR и MAD
Для высоты: Q1−1,5IQR и Q3+1,5IQR; модифицированное z-значение по MAD больше 3,5. Флаги не доказывают ошибку измерения.

In [10]:
df, stats = apply_step(df, STEPS[8])
audit.append(stats)
display(pd.DataFrame([stats]))
display(df.head(2))

,stage,processed,after,deleted,changed,iqr_low,iqr_high,median,mad,iqr,mad_count,both
0,outliers,1600000,1600000,0,205990,-36.0,60.0,7.0,9.0,186965,205564,186539


,id,type,language,license,rights_holder,access_rights,dataset_id,institution_code,dataset_name,owner_institution_code,...,scientific_name,kingdom,taxon_rank,vernacular_name,height_missing,sex_missing,height_negative,seconds_since_previous,height_iqr_flag,height_mad_flag
0,801:20130517200539,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,Larus argentatus,Animalia,species,Herring Gull,False,False,False,1815,False,False
1,801:20130517223523,Event,en,http://creativecommons.org/publicdomain/zero/1.0/,INBO,http://www.inbo.be/en/norms-for-data-use,https://doi.org/10.15468/02omly,INBO,Bird tracking - GPS tracking of Lesser Black-b...,INBO/VLIZ/UGent/UA,...,Larus argentatus,Animalia,species,Herring Gull,False,False,False,1790,False,False


## Проверка результата
Сравнение построчное, вместе с типами. Повторная очистка должна сохранить таблицу.

In [11]:
from src.validation import assert_same
df.to_parquet(PROCESSED / "clean_notebook.parquet", index=False)
assert_same(df, pd.read_parquet(PROCESSED / "clean.parquet"))
again = df.copy()
for step in STEPS:
    again, _ = apply_step(again, step)
assert_same(df, again)
print("Блокнот = CLI; повторная очистка = исходный результат")
display(pd.DataFrame(audit))

Блокнот = CLI; повторная очистка = исходный результат


,stage,processed,after,deleted,changed,note,parse_failures,full_duplicates,key_duplicates,invalid_domain,negative_height,missing_interval,iqr_low,iqr_high,median,mad,iqr,mad_count,both
0,names,1600000,1600000,0,0,Переименованы столбцы,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,assign_types,1600000,1600000,0,0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,hidden_missing,1600000,1600000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,missing,1600000,1600000,0,201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,duplicates,1600000,1600000,0,0,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,categories,1600000,1600000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,text_field,1600000,1600000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,constraints,1600000,1600000,0,363717,NaN,NaN,NaN,NaN,0.0,363699.0,22.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,outliers,1600000,1600000,0,205990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-36.0,60.0,7.0,9.0,186965.0,205564.0,186539.0
